# PSS Augmentation with CKN — Tutorial

Walkthrough of the `skm_tools.augment_pss` module. Three parts:

- **PART ONE** — Build (or load) an augmented PSS network: every CKN node/edge that touches PSS is added, tagged `origin="CKN"`; PSS itself is untouched and always tagged `origin="PSS"`.
- **PART TWO** — Inspect what the augmentation introduced:
  - **Export A** — PSS pairs now directly connected by a new CKN edge
  - **Export B** — PSS pairs now connected through exactly one CKN intermediate
- **PART THREE** — Pick any node (random or chosen) and check how much CKN augmented its immediate neighbourhood.

Run this notebook top to bottom in VS Code (with the Jupyter extension) using a kernel that has `networkx`, `pandas`, and (for the visualization cells) `py4cytoscape` with Cytoscape running locally.

## Setup

In [1]:
from pathlib import Path
from datetime import date
import importlib
import networkx as nx
import pandas as pd

from skm_tools import load_networks, augment_pss, ckn_utils
importlib.reload(augment_pss)   # pick up edits without restarting the kernel

base_dir = Path("./")
data_dir = base_dir / "data"
output_dir = base_dir / "output"
data_dir.mkdir(exist_ok=True)
output_dir.mkdir(exist_ok=True)

print("Imports OK")

Imports OK


---
## PART ONE — Build (or load) the augmented PSS network

`build_augmented_pss(pss, ckn)`:
1. Copies PSS as-is (`origin="PSS"` on every node/edge)
2. Adds every CKN edge that touches at least one PSS node, and the CKN node(s) it brings in (`origin="CKN"`) — unless PSS already has an edge between the same two nodes (PSS always wins)
3. Adds CKN-CKN edges between two CKN nodes that were both pulled in by step 2 (keeps local CKN connectivity intact, needed for Export B and for node-neighbourhood inspection in PART THREE)

No rank filter is applied here — every CKN rank is kept, so the saved network is reusable for any rank threshold later.

In [2]:
# ---- USER INPUT ------------------------------------------------------------
AUGMENTED_DIR = output_dir / "augmented_pss"
REBUILD = False   # set True to force a rebuild even if saved files exist
# ---------------------------------------------------------------------------

today = date.today().isoformat()

pss = load_networks.pss_dinar_to_networkx(
    edge_path=data_dir / f"pss-dinar-edges-public-{today}.tsv",
    node_path=data_dir / f"pss-dinar-nodes-public-{today}.tsv",
)
print(f"PSS: {pss.number_of_nodes():,} nodes, {pss.number_of_edges():,} edges")

graphml_path = AUGMENTED_DIR / "augmented.graphml"
if graphml_path.exists() and not REBUILD:
    print(f"\nFound saved augmented graph at {graphml_path} -- loading.")
    augmented = augment_pss.load_augmented(AUGMENTED_DIR)
else:
    print("\nBuilding augmented graph from scratch ...")
    ckn = load_networks.ckn_to_networkx(
        edge_path=data_dir / "AtCKN-v2-2023.06.tsv.gz",
        node_path=data_dir / "AtCKN-v2-2023.06_node-annot.tsv.gz",
        add_reciprocal_edges=True,
    )
    ckn_utils.filter_ckn_nodes(ckn, species=['ath'])
    print(f"CKN: {ckn.number_of_nodes():,} nodes, {ckn.number_of_edges():,} edges")

    augmented = augment_pss.build_augmented_pss(pss, ckn, verbose=True)
    augment_pss.save_augmented(augmented, AUGMENTED_DIR)

print(f"\nAugmented: {augmented.number_of_nodes():,} nodes, "
      f"{augmented.number_of_edges():,} edges")

PSS: 1,071 nodes, 4,395 edges

Building augmented graph from scratch ...
Removed 162 nodes from network.
CKN: 26,072 nodes, 898,049 edges
[1/3] Copying PSS nodes and edges …
      PSS: 1,071 nodes, 4,395 edges
[2/3] Adding CKN edges that touch PSS …
      + 11,917 CKN nodes, 153,183 CKN edges (touching PSS)
[3/3] Adding CKN edges between pulled-in CKN nodes …
      + 557,277 CKN-CKN edges
----------------------------------------------------------
  Augmented: 12,988 nodes (1,071 PSS + 11,917 CKN)
  Augmented: 714,855 edges (4,395 PSS + 710,460 CKN)
----------------------------------------------------------

OK: Saved augmented network to output\augmented_pss/
  augmented.graphml      (12,988 nodes, 714,855 edges)
  augmented_nodes.tsv    (12,988 rows)
  augmented_edges.tsv    (714,855 rows)
  augmentation_diff.tsv   (722,377 rows: 11,917 CKN nodes + 710,460 CKN edges)

Augmented: 12,988 nodes, 714,855 edges


### Before / after summary

In [3]:
_ = augment_pss.print_augmentation_summary(pss, augmented)

                          PSS (before)  Augmented (after)
--------------------------------------------------
                   Nodes         1,071             12,988
      of which CKN-added             —             11,917
                   Edges         4,395            714,855
      of which CKN-added             —            710,460


### Unaugmented nodes

`build_augmented_pss()` automatically calls `check_unaugmented_nodes()` and
prints a warning listing any PSS node that gained no new CKN connections
(e.g. it isn't present in CKN, or every potential CKN edge was skipped
because PSS already had an edge there). This is especially useful for
small, hand-built subnetworks where a node may be mistyped.

The check can also be re-run manually on any `(pss, augmented)` pair.

In [ ]:
unaugmented = augment_pss.check_unaugmented_nodes(pss, augmented)
print(f"\n{len(unaugmented)} unaugmented node(s) out of {pss.number_of_nodes()} PSS nodes.")

### Visualize a small sample in Cytoscape

The full augmented network is too large to render usefully. We take the
ego-graph (radius 1) around a few PSS nodes to visually confirm PSS
(blue) vs CKN (orange) colouring.

In [ ]:
pss_nodes = [n for n, d in augmented.nodes(data=True) if d.get("origin") == "PSS"]
sample_seed_nodes = pss_nodes[:2]

sample_nodes = set(sample_seed_nodes)
for n in sample_seed_nodes:
    sample_nodes |= set(nx.ego_graph(augmented.to_undirected(), n, radius=1).nodes())

sample = augmented.subgraph(sample_nodes).copy()
print(f"Sample subgraph: {sample.number_of_nodes()} nodes, {sample.number_of_edges()} edges")

suid_sample = augment_pss.visualize_augmented_network(sample, network_name="augmented_pss_sample")

Sample subgraph: 114 nodes, 1254 edges
Sending to Cytoscape: 114 nodes, 1254 edges …
Applying default style...
Applying preferred layout
Applied CKN-default to 42295
✓ Network 'augmented_pss_sample' created in Cytoscape (SUID: 42295)


---
## PART TWO — Augmentation analysis

Set the rank threshold applied here (analysis time, not build time).

In [ ]:
# ---- USER INPUT ------------------------------------------------------------
MAX_RANK = 1   # 0 = highest confidence only, None = keep all ranks
# ---------------------------------------------------------------------------

### Export A — new direct CKN edges between PSS pairs

In [ ]:
export_a = augment_pss.find_new_direct_edges(augmented, pss, max_rank=MAX_RANK)
print(f"Export A: {len(export_a):,} new direct-edge PSS pairs (rank <= {MAX_RANK})")
export_a.head(20)

Export A: 830 new direct-edge PSS pairs (rank <= 1)


,u,v,rank,direction
0,AT1G08320,AT2G14580,0,u->v
1,AT1G08320,AT2G14610,0,u->v
2,AT1G08450,AT1G04310,0,both
3,AT1G08450,AT1G66340,0,both
4,AT1G08450,AT2G40940,0,both
5,AT1G08450,AT3G04580,0,both
6,AT1G08450,AT3G23150,0,both
7,AT1G09210,AT1G04310,0,both
8,AT1G09210,AT1G66340,0,both
9,AT1G09210,AT2G40940,0,both


In [ ]:
#AT2G26150, AT3G12580

In [ ]:
export_a.to_csv(output_dir / "export_A_new_direct_edges.csv", index=False)
print("Saved -> output/export_A_new_direct_edges.csv")

Saved -> output/export_A_new_direct_edges.csv


### Export B — PSS pairs via exactly one CKN intermediate

In [ ]:
export_b = augment_pss.find_new_intermediate_pairs(augmented, pss, max_rank=MAX_RANK)
n_unique_pairs = export_b[["u", "v"]].drop_duplicates().shape[0] if len(export_b) else 0
print(f"Export B: {len(export_b):,} pair-intermediate triples, {n_unique_pairs:,} unique pairs (rank <= {MAX_RANK})")
export_b.head(20)

Export B: 4,282 pair-intermediate triples, 2,277 unique pairs (rank <= 1)


,u,v,intermediate,rank_ux,rank_xv,path_rank
0,AT1G04100,AT1G04240,AUX/IAA|WRKY57,0,0,0
1,AT1G04100,AT1G04250,AUX/IAA|WRKY57,0,0,0
2,AT1G04100,AT1G04550,AUX/IAA|WRKY57,0,0,0
3,AT1G04100,AT1G15050,AUX/IAA|WRKY57,0,0,0
4,AT1G04100,AT1G15580,AUX/IAA|WRKY57,0,0,0
5,AT1G04100,AT1G51950,AUX/IAA|WRKY57,0,0,0
6,AT1G04100,AT1G52830,AUX/IAA|WRKY57,0,0,0
7,AT1G04100,AT1G80390,AUX/IAA|WRKY57,0,0,0
8,AT1G04100,AT2G01200,AUX/IAA|WRKY57,0,0,0
9,AT1G04100,AT2G22670,AUX/IAA|WRKY57,0,0,0


In [ ]:
export_b.to_csv(output_dir / "export_B_one_intermediate_pairs.csv", index=False)
print("Saved -> output/export_B_one_intermediate_pairs.csv")

Saved -> output/export_B_one_intermediate_pairs.csv


### Inspect one example pair from each export in Cytoscape

In [ ]:
# ---- USER INPUT ------------------------------------------------------------
PAIR_A_INDEX = 0
PAIR_B_INDEX = 0
MAX_NEIGHBOURS = 10
# ---------------------------------------------------------------------------

In [ ]:
if len(export_a):
    row_a = export_a.iloc[PAIR_A_INDEX]
    print(f"Export A pair: {row_a.u} <-> {row_a.v}  (rank {row_a['rank']}, direction {row_a.direction})")

    sub_a = augment_pss.extract_pair_subnetwork(
        augmented, pss, a=row_a.u, b=row_a.v, intermediate=None,
        max_neighbours=MAX_NEIGHBOURS,
    )
    print(f"  Subnetwork: {sub_a.number_of_nodes()} nodes, {sub_a.number_of_edges()} edges")
    suid_a = augment_pss.visualize_pair_subnetwork(sub_a, network_name=f"expA_{row_a.u}_{row_a.v}")
else:
    print("Export A is empty at this rank threshold.")

Export A pair: AT1G08320 <-> AT2G14580  (rank 0, direction u->v)
  Subnetwork: 22 nodes, 133 edges
Sending to Cytoscape: 22 nodes, 133 edges …
Applying default style...
Applying preferred layout
Applied CKN-default to 49401
✓ Network 'expA_AT1G08320_AT2G14580' created in Cytoscape (SUID: 49401)


In [ ]:
if len(export_b):
    row_b = export_b.iloc[PAIR_B_INDEX]
    print(f"Export B triple: {row_b.u} <- {row_b.intermediate} -> {row_b.v}  "
          f"(rank_ux={row_b.rank_ux}, rank_xv={row_b.rank_xv}, path={row_b.path_rank})")

    sub_b = augment_pss.extract_pair_subnetwork(
        augmented, pss, a=row_b.u, b=row_b.v, intermediate=row_b.intermediate,
        max_neighbours=MAX_NEIGHBOURS,
    )
    print(f"  Subnetwork: {sub_b.number_of_nodes()} nodes, {sub_b.number_of_edges()} edges")
    suid_b = augment_pss.visualize_pair_subnetwork(
        sub_b, network_name=f"expB_{row_b.u}_{row_b.intermediate}_{row_b.v}")
else:
    print("Export B is empty at this rank threshold.")

Export B triple: AT1G04100 <- AUX/IAA|WRKY57 -> AT1G04240  (rank_ux=0, rank_xv=0, path=0)
  Subnetwork: 9 nodes, 37 edges
Sending to Cytoscape: 9 nodes, 37 edges …
Applying default style...
Applying preferred layout
Applied CKN-default to 50350
✓ Network 'expB_AT1G04100_AUX/IAA|WRKY57_AT1G04240' created in Cytoscape (SUID: 50350)


---
## PART THREE — Node neighbourhood inspection

Pick any node (random, or one you choose) and check how much CKN augmented
its immediate neighbourhood.

**How big should the neighbourhood be?**
- **`radius=1`** (default, recommended) — direct neighbours only. Answers
  "did CKN change who this exact node talks to?" Always small enough to
  read directly.
- **`radius=2`** — neighbours-of-neighbours. Useful for spotting pathway
  crosstalk opened up by a CKN intermediate, but some CKN hub nodes bridge
  hundreds to thousands of PSS pairs (see PART TWO Export B), so a radius-2
  view can blow up. Always combine with `max_neighbours` (try 15-25) when
  using radius 2.
- **`radius>=3`** is not recommended for visually inspecting a single node
  — use the Export A/B tables from PART TWO for that instead.

In [ ]:
# ---- USER INPUT ------------------------------------------------------------
NODE = None        # set to a specific AGI code / CKN id, or leave None for random
NODE_TYPE = "PSS"  # "PSS", "CKN", or None -- only used when NODE is None
SEED = None         # set an int for a reproducible random pick
RADIUS = 1
MAX_NEIGHBOURS = 20  # recommended whenever RADIUS > 1
# ---------------------------------------------------------------------------

node = NODE or augment_pss.pick_random_node(augmented, node_type=NODE_TYPE, seed=SEED)
print(f"Inspecting node: {node}  (origin={augmented.nodes[node].get('origin')})")

Inspecting node: AT4G24650  (origin=PSS)


In [2]:
stats = augment_pss.get_node_neighbourhood_augmentation(
    pss, augmented, node, radius=RADIUS, max_neighbours=MAX_NEIGHBOURS
)
augment_pss.print_node_neighbourhood_summary(stats)

NameError: name 'augment_pss' is not defined

In [2]:
import skm_tools.augment_pss as augment_pss
print(augment_pss.__file__)
print(hasattr(augment_pss, 'get_node_neighbourhood_augmentation'))

ModuleNotFoundError: No module named 'skm_tools'

In [17]:
import skm_tools.augment_pss as augment_pss
print(augment_pss.__file__)
print(hasattr(augment_pss, 'get_node_neighbourhood_augmentation'))

c:\users\vidm\desktop\skm_ckn\skm_tools\augment_pss.py
False


In [16]:
neighbourhood = augment_pss.extract_node_neighbourhood(
    augmented, node, radius=RADIUS, max_neighbours=MAX_NEIGHBOURS
)
suid_node = augment_pss.visualize_node_neighbourhood(
    neighbourhood, node, network_name=f"neighbourhood_{node}_r{RADIUS}"
)

AttributeError: module 'skm_tools.augment_pss' has no attribute 'extract_node_neighbourhood'

---
## PART FOUR — Augment your own hand-built subnetwork

You don't have to augment the full PSS network. `build_augmented_pss()` takes
any NetworkX graph as `pss` — including a small graph you build by hand for
genes you're interested in, or one imported from another tool (e.g. a
selection made in Cytoscape).

Below, one node is a deliberate typo/placeholder that doesn't exist in PSS
or CKN — `check_unaugmented_nodes()` catches it automatically instead of
failing silently.

In [ ]:
if 'ckn' not in dir():
    ckn = load_networks.ckn_to_networkx(
        edge_path=data_dir / "AtCKN-v2-2023.06.tsv.gz",
        node_path=data_dir / "AtCKN-v2-2023.06_node-annot.tsv.gz",
        add_reciprocal_edges=True,
    )
    ckn_utils.filter_ckn_nodes(ckn, species=['ath'])

# A small network of interest, built by hand (not derived from the full PSS)
custom_pss = nx.MultiDiGraph()
custom_pss.add_node("AT5G12080")
custom_pss.add_node("AT3G02090")
custom_pss.add_node("NOT_A_REAL_GENE_ID")   # e.g. a typo'd AGI code
custom_pss.add_edge("AT5G12080", "AT3G02090")

custom_augmented = augment_pss.build_augmented_pss(custom_pss, ckn, verbose=True)
_ = augment_pss.print_augmentation_summary(custom_pss, custom_augmented)

### Analyse the custom augmented subnetwork

The PART TWO/THREE analysis functions (`find_new_direct_edges`,
`find_new_intermediate_pairs`, node-neighbourhood inspection) don't care
whether `augmented` came from the full PSS network or a small hand-built
one — pass `custom_augmented`/`custom_pss` in exactly the same way.

In [ ]:
custom_export_a = augment_pss.find_new_direct_edges(custom_augmented, custom_pss)
print(f"Export A: {len(custom_export_a):,} new direct-edge pairs")
if len(custom_export_a):
    print(custom_export_a)

custom_export_b = augment_pss.find_new_intermediate_pairs(custom_augmented, custom_pss)
print(f"\nExport B: {len(custom_export_b):,} pair-intermediate triples")
if len(custom_export_b):
    print(custom_export_b)

# Node-neighbourhood inspection also works unchanged on the custom result
custom_node = augment_pss.pick_random_node(custom_augmented, node_type="PSS", seed=0)
custom_stats = augment_pss.get_node_neighbourhood_augmentation(
    custom_pss, custom_augmented, custom_node, radius=1
)
print()
augment_pss.print_node_neighbourhood_summary(custom_stats)

---
## Summary

In [16]:
s = augment_pss.get_augmentation_summary(pss, augmented)
n_unique_pairs = export_b[["u", "v"]].drop_duplicates().shape[0] if len(export_b) else 0

print(f"""
==================================================================
  PSS augmentation summary  (rank <= {MAX_RANK})
==================================================================

  Augmented graph
    Nodes : {s['augmented_nodes']:>7,}  ({s['pss_nodes']:,} PSS + {s['ckn_nodes_added']:,} CKN)
    Edges : {s['augmented_edges']:>7,}  ({s['pss_edges']:,} PSS + {s['ckn_edges_added']:,} CKN)

  Export A -- new direct CKN edges between PSS pairs
    {len(export_a):>7,} pairs -> output/export_A_new_direct_edges.csv

  Export B -- 1-CKN-intermediate pairs
    {len(export_b):>7,} (pair, intermediate) triples
    {n_unique_pairs:>7,} unique PSS pairs -> output/export_B_one_intermediate_pairs.csv

  Part Three -- inspected node
    {stats['node']}  radius={stats['radius']}
    {stats['ckn_added_neighbours']} of {stats['augmented_neighbours']} neighbours are new from CKN
    ({stats['pct_new']:.1f}%)

  Files in {AUGMENTED_DIR}/
    augmented.graphml, augmented_nodes.tsv, augmented_edges.tsv, augmentation_diff.tsv
==================================================================
""")


  PSS augmentation summary  (rank <= 1)

  Augmented graph
    Nodes :  13,024  (1,071 PSS + 11,953 CKN)
    Edges : 715,125  (4,395 PSS + 710,730 CKN)

  Export A -- new direct CKN edges between PSS pairs
        836 pairs -> output/export_A_new_direct_edges.csv

  Export B -- 1-CKN-intermediate pairs
      6,586 (pair, intermediate) triples
      4,498 unique PSS pairs -> output/export_B_one_intermediate_pairs.csv

  Part Three -- inspected node
    AT4G31800  radius=1
    15 of 20 neighbours are new from CKN
    (75.0%)

  Files in output\augmented_pss/
    augmented.graphml, augmented_nodes.tsv, augmented_edges.tsv, augmentation_diff.tsv

